# Long-lived notebook connection

Connects to a running backend, stays **idle** (no calls at all) while other clients
work with the instruments, then takes control.

Used as an automated check (environment variables `EM_MIN_IDLE_S` and `EM_DONE_FILE`
set by the test harness), but it also works interactively: start a backend with
`python -m easymeasure.backend --setup examples/faulty_setup.json`, then run the cells.
Without the variables, the idle wait is skipped.

In [ ]:
import os, time, collections
import numpy as np
from easymeasure.client import BackendClient
from easymeasure.backend.storage import DataSet

c = BackendClient.connect(name="notebook")          # default: status events only
events = collections.Counter()
c.on_event = lambda kind, data: events.update([kind])
local_addr = c.sock.getsockname()
t_connect = time.time()
print("connected to backend pid", c.welcome["pid"], "from", local_addr)

## Stay idle while others work
No request is sent during this wait; the connection is just open.

In [ ]:
min_idle = float(os.environ.get("EM_MIN_IDLE_S", "0"))
done_file = os.environ.get("EM_DONE_FILE")
while True:
    idle = time.time() - t_connect
    other_done = done_file is None or os.path.exists(done_file)
    if idle >= min_idle and other_done:
        break
    time.sleep(5)
print("idle for %.1f min; status events received meanwhile: %s" % (idle / 60, dict(events)))

## Still the same connection?

In [ ]:
assert not c.closed, "connection was closed"
assert c.sock.getsockname() == local_addr          # same TCP connection as at the start
t = time.time(); c.call("ping"); print("ping %.1f ms" % (1000 * (time.time() - t)))
assert not ({"monitor", "sweep_point", "log"} & set(events)), "got data it did not subscribe to"
state = c.call("queue_state")
print("queue:", state["state"], "| items run by the other client:", len(state["history"]))
assert state["state"] == "idle" and not state["queued"]

## Take control of the hardware

In [ ]:
def retry(fn, n=5):
    # the simulated hardware drops / garbles commands on purpose: a script talking to
    # flaky hardware retries at its level too, like a person would press the button again
    for i in range(n):
        try:
            return fn()
        except Exception as exc:
            last = exc
            time.sleep(0.2)
    raise last

for target in (0.25, -0.15, 0.0):
    retry(lambda: c.call("write_channel", name="Vg", value=target))      # protected + ramped
    v = retry(lambda: c.call("read_channel", name="Vg"))
    print("Vg set %+.3f read %+.4f" % (target, v))
retry(lambda: c.call("write_channel", name="dmm.nplc", value=5))           # config channel
print("dmm.nplc =", retry(lambda: c.call("read_channel", name="dmm.nplc")))
for act in ("dmm.reset", "lockin.auto_gain", "awg.start", "awg.stop"):
    retry(lambda: c.call("write_channel", name=act))
print("dmm.nplc after reset =", retry(lambda: c.call("read_channel", name="dmm.nplc")))
try:
    c.call("write_channel", name="Vg", value=50)
    raise AssertionError("limit not enforced")
except Exception as exc:
    print("out-of-limit write refused:", type(exc).__name__)

In [ ]:
sweep = {"name": "notebook_takeover", "on_read_error": "nan", "dimensions": [
    {"points": 31, "sweep": [{"channel": "Vg", "start": -0.4, "stop": 0.4}], "record": ["I", "X"]},
    {"points": 6, "sweep": [{"channel": "Vb", "start": -0.2, "stop": 0.2}], "record": ["T"]}],
    "pre": {"hook": "set('dmm.reset')", "record": ["dmm.nplc"]},
    "post": {"constants": [{"channel": "Vg", "value": 0.0}, {"channel": "Vb", "value": 0.0}]}}
item = c.call("queue_add", item={"kind": "sweep", "definition": sweep}, start=True)
while True:
    st = c.call("queue_state")
    if st["state"] == "idle" and st["history"] and st["history"][-1]["id"] == item:
        break
    time.sleep(0.5)
res = st["history"][-1]
ds = DataSet.load(res["path"])
good = np.isfinite(ds.data["I"]).mean()
print("sweep", res["status"], "| points %d/%d | finite I %.0f%%" % (ds.points, ds.total_points, 100 * good))
assert res["status"] == "finished" and ds.points == ds.total_points and good > 0.8
print("status events received in total:", dict(events))

In [ ]:
print("connected for %.1f min, same connection: %s" % ((time.time() - t_connect) / 60,
      c.sock.getsockname() == local_addr))
c.close()